In [1]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.3 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, random_split

def mi_edge_index(mi_dict_path, top_k=6, return_edge_attr=False):
    """
    mi_dict: {col_name: pd.Series} (index=다른 변수명, value=MI, 내림차순 정렬)
    top_k: 각 src에서 MI 상위 k개로만 유향 엣지(src->dst) 생성
    return_edge_attr: True면 edge_attr로 MI 가중치 반환
    """

    import pickle
    with open(mi_dict_path, 'rb') as f:
        mi_dict = pickle.load(f)

    cols = list(mi_dict.keys())
    col_to_idx = {c: i for i, c in enumerate(cols)}

    src_idx, dst_idx, weights = [], [], []

    for src in cols:
        series = mi_dict[src]

        # 우리 그래프에 존재하는 변수만 남기고, 자기 자신은 제외
        series = series[series.index.isin(cols)]
        if src in series.index:
            series = series.drop(index=src)

        # 상위 k개 선택 (이미 내림차순 정렬되어 있다고 가정)
        top_neighbors = series.head(top_k)

        for dst, w in top_neighbors.items():
            src_idx.append(col_to_idx[src])
            dst_idx.append(col_to_idx[dst])
            if return_edge_attr:
                weights.append(float(w))

    edge_index = torch.tensor([src_idx, dst_idx], dtype=torch.long)

    if return_edge_attr:
        edge_attr = torch.tensor(weights, dtype=torch.float)
        return edge_index, edge_attr

    return edge_index

def mi_edge_index_batched(batch_size, mi_dict_path, top_k=6, return_edge_attr=False)->torch.Tensor:
    single = mi_edge_index(mi_dict_path=mi_dict_path, top_k=top_k, return_edge_attr=return_edge_attr)

    batch_list = [single for _ in range(batch_size)]

    return torch.concatenate(tensors=batch_list, dim=1) # type: ignore

def get_col_dims(df: pd.DataFrame):
    '''
    변수별 범주의 개수 파악
    '''
    col_dims = [len(df[col].unique()) for col in df.columns]
    return col_dims

def get_ad_dis_col(df:pd.DataFrame):
    '''
    admission 시의 컬럼, discharge 시의 컬럼을 나누어 리턴
    Args:
        df(pd.DataFrame): 원본 데이터프레임, REASONb는 자동으로 제외됨
    Returns:
        (admission 시의 컬럼 list, discharge 시의 컬럼 list)
    '''
    cols = list(df.columns)

    if 'LOS' in cols:
        cols.remove('LOS')

    if 'REASONb' in cols:
        cols.remove('REASONb')

    change = []
    change_D = []

    for i in cols:
        if i.endswith('_D'):
            change_D.append(i)
            change.append(i[:-2])

    ad = [i for i in cols if i not in change_D]
    dis = ad.copy()
    for i in range(len(ad)):
        if dis[i] in change:
            dis[i] = dis[i] + '_D'

    return ad, dis

def find_indices(lst, targets):
    return [lst.index(t) if t in lst else None for t in targets]

def get_ad_dis_index(df: pd.DataFrame):
    col_list = list(df.columns)
    ad, dis = get_ad_dis_col(df)
    ad_col_index = find_indices(col_list, ad)
    dis_col_index = find_indices(col_list, dis)
    return ad_col_index, dis_col_index

def get_col_info(df: pd.DataFrame):
    '''
    Returns: (tuple)
        col_list, col_dims, ad_col_index, dis_col_index

        col_list: 보관용, 데이터에 등장하는 열 이름의 순서
        col_dims: col_list 순서대로 변수별 범주의 개수
        ad_col_index: admission에 해당하는 변수의 integer position
        dis_col_index: discharge에 해당하는 변수의 integer position
    '''
    col_list = list(df.columns)
    col_dims = get_col_dims(df)
    ad_col_index, dis_col_index = get_ad_dis_index(df)
    return col_list, col_dims, ad_col_index, dis_col_index

def organize_labels(df: pd.DataFrame):
    '''
    -9가 있는 변수를 그대로 엔티티 임베딩에 넣으면 이상해짐
    왜냐하면 엔티티 임베딩 모델은 레이블들이 연속된 정수들의 범위로 있다고 가정하기 때문
    -9, 1, 2, 3 이렇게 있었다면
    -9, -8, -7, -6, -5, ~~~ 이런 것으로 가정함

    -9, 1, 2, 3를
    0, 1, 2, 3으로 바꿈 (-9 -> 4)

    + CBSA2020
    이것도 문제가 됨
    10000 24242 32646 75577 이런 식이라 연속된 정수들의 레이블이 아님
    10000 24242 32646 75577 -> 1, 2, 3, 4
    '''

    for col in df.columns:
        labels = sorted(df[col].unique())
        replace_dict = {labels[i]: i for i in range(len(labels))}
        df[col] = df[col].replace(replace_dict)

    return df

def df_to_tensor(df: pd.DataFrame | pd.Series, dtype=torch.long):
    df_np = df.to_numpy()
    return torch.tensor(df_np, dtype=dtype)

def get_total_dim(df: pd.DataFrame):
    total_dim = 0
    for col in df.columns:
        col_dim = len(df[col].unique())
        total_dim += col_dim
    return total_dim


def train_test_split_customed(dataset, batch_size, ratio=[0.7, 0.15, 0.15], seed=42, num_workers=0):

    train_dataset, val_dataset, test_dataset = random_split(
        dataset=dataset,
        lengths=ratio,
        generator=torch.Generator().manual_seed(seed)
    )

    print(f"Train Set Size: {len(train_dataset)}")
    print(f"Test Set Size: {len(test_dataset)}")

    train_dataloader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_dataloader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    test_dataloader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

    return train_dataloader, val_dataloader, test_dataloader

In [4]:
import torch
from torch_geometric.nn import GCNConv




class TGCN2(torch.nn.Module):
    r"""An implementation THAT SUPPORTS BATCHES of the Temporal Graph Convolutional Gated Recurrent Cell.
    For details see this paper: `"T-GCN: A Temporal Graph ConvolutionalNetwork for
    Traffic Prediction." <https://arxiv.org/abs/1811.05320>`_

    Args:
        in_channels (int): Number of input features.
        out_channels (int): Number of output features.
        batch_size (int): Size of the batch.
        improved (bool): Stronger self loops. Default is False.
        cached (bool): Caching the message weights. Default is False.
        add_self_loops (bool): Adding self-loops for smoothing. Default is True.
    """

    def __init__(self, in_channels: int, out_channels: int,
                 batch_size: int,  # this entry is unnecessary, kept only for backward compatibility
                 improved: bool = False, cached: bool = False,
                 add_self_loops: bool = True):
        super(TGCN2, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.improved = improved
        self.cached = cached
        self.add_self_loops = add_self_loops
        self.batch_size = batch_size  # not needed
        self._create_parameters_and_layers()

    def _create_update_gate_parameters_and_layers(self):
        self.conv_z = GCNConv(in_channels=self.in_channels,  out_channels=self.out_channels, improved=self.improved,
                              cached=self.cached, add_self_loops=self.add_self_loops )
        self.linear_z = torch.nn.Linear(2 * self.out_channels, self.out_channels)

    def _create_reset_gate_parameters_and_layers(self):
        self.conv_r = GCNConv(in_channels=self.in_channels, out_channels=self.out_channels, improved=self.improved,
                              cached=self.cached, add_self_loops=self.add_self_loops )
        self.linear_r = torch.nn.Linear(2 * self.out_channels, self.out_channels)

    def _create_candidate_state_parameters_and_layers(self):
        self.conv_h = GCNConv(in_channels=self.in_channels, out_channels=self.out_channels, improved=self.improved,
                              cached=self.cached, add_self_loops=self.add_self_loops )
        self.linear_h = torch.nn.Linear(2 * self.out_channels, self.out_channels)

    def _create_parameters_and_layers(self):
        self._create_update_gate_parameters_and_layers()
        self._create_reset_gate_parameters_and_layers()
        self._create_candidate_state_parameters_and_layers()

    def _set_hidden_state(self, X, H):
        if H is None:
            # can infer batch_size from X.shape, because X is [B, N, F]
            H = torch.zeros(X.shape[0], X.shape[1], self.out_channels).to(X.device) #(b, 207, 32)
        return H

    def _calculate_update_gate(self, X, edge_index, edge_weight, H):
        Z = torch.cat([self.conv_z(X, edge_index, edge_weight), H], axis=2) # (b, 207, 64)
        Z = self.linear_z(Z) # (b, 207, 32)
        Z = torch.sigmoid(Z)

        return Z

    def _calculate_reset_gate(self, X, edge_index, edge_weight, H):
        R = torch.cat([self.conv_r(X, edge_index, edge_weight), H], axis=2) # (b, 207, 64)
        R = self.linear_r(R) # (b, 207, 32)
        R = torch.sigmoid(R)

        return R

    def _calculate_candidate_state(self, X, edge_index, edge_weight, H, R):
        H_tilde = torch.cat([self.conv_h(X, edge_index, edge_weight), H * R], axis=2) # (b, 207, 64)
        H_tilde = self.linear_h(H_tilde) # (b, 207, 32)
        H_tilde = torch.tanh(H_tilde)

        return H_tilde

    def _calculate_hidden_state(self, Z, H, H_tilde):
        H = Z * H + (1 - Z) * H_tilde   # # (b, 207, 32)
        return H

    def forward(self,X: torch.FloatTensor, edge_index: torch.LongTensor, edge_weight: torch.FloatTensor = None,
                H: torch.FloatTensor = None ) -> torch.FloatTensor:
        """
        Making a forward pass. If edge weights are not present the forward pass
        defaults to an unweighted graph. If the hidden state matrix is not present
        when the forward pass is called it is initialized with zeros.

        Arg types:
            * **X** *(PyTorch Float Tensor)* - Node features.
            * **edge_index** *(PyTorch Long Tensor)* - Graph edge indices.
            * **edge_weight** *(PyTorch Long Tensor, optional)* - Edge weight vector.
            * **H** *(PyTorch Float Tensor, optional)* - Hidden state matrix for all nodes.

        Return types:
            * **H** *(PyTorch Float Tensor)* - Hidden state matrix for all nodes.
        """
        H = self._set_hidden_state(X, H)
        Z = self._calculate_update_gate(X, edge_index, edge_weight, H)
        R = self._calculate_reset_gate(X, edge_index, edge_weight, H)
        H_tilde = self._calculate_candidate_state(X, edge_index, edge_weight, H, R)
        H = self._calculate_hidden_state(Z, H, H_tilde) # (b, 207, 32)
        return H

In [5]:
import torch
import sys
import os

class A3TGCN2(torch.nn.Module):
    r"""An implementation THAT SUPPORTS BATCHES of the Attention Temporal Graph Convolutional Cell.
    For details see this paper: `"A3T-GCN: Attention Temporal Graph Convolutional
    Network for Traffic Forecasting." <https://arxiv.org/abs/2006.11583>`_

    Args:
        in_channels (int): Number of input features.
        out_channels (int): Number of output features.
        periods (int): Number of time periods.
        improved (bool): Stronger self loops (default :obj:`False`).
        cached (bool): Caching the message weights (default :obj:`False`).
        add_self_loops (bool): Adding self-loops for smoothing (default :obj:`True`).
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        periods: int,
        batch_size:int,
        improved: bool = False,
        cached: bool = False,
        add_self_loops: bool = True):
        super(A3TGCN2, self).__init__()

        self.in_channels = in_channels  # 2
        self.out_channels = out_channels # 32
        self.periods = periods # 12
        self.improved = improved
        self.cached = cached
        self.add_self_loops = add_self_loops
        self.batch_size = batch_size
        self._setup_layers()

    def _setup_layers(self):
        self._base_tgcn = TGCN2(
            in_channels=self.in_channels,
            out_channels=self.out_channels,
            batch_size=self.batch_size,
            improved=self.improved,
            cached=self.cached,
            add_self_loops=self.add_self_loops)

        device = device_set()
        self._attention = torch.nn.Parameter(torch.empty(self.periods, device=device))
        torch.nn.init.uniform_(self._attention)

    def forward(
        self,
        X: torch.FloatTensor,
        edge_index: torch.LongTensor,
        edge_weight: torch.FloatTensor | None = None,
        H: torch.FloatTensor | None  = None
    ) -> torch.FloatTensor:
        """
        Making a forward pass. If edge weights are not present the forward pass
        defaults to an unweighted graph. If the hidden state matrix is not present
        when the forward pass is called it is initialized with zeros.

        Arg types:
            * **X** (PyTorch Float Tensor): Node features for T time periods.
            * **edge_index** (PyTorch Long Tensor): Graph edge indices.
            * **edge_weight** (PyTorch Long Tensor, optional)*: Edge weight vector.
            * **H** (PyTorch Float Tensor, optional): Hidden state matrix for all nodes.

        Return types:
            * **H** (PyTorch Float Tensor): Hidden state matrix for all nodes.
        """
        H_accum = 0
        H_sequence_outputs = []

        probs = torch.nn.functional.softmax(self._attention, dim=0)

        H_previous = H

        for period in range(self.periods):
            X_current = X[:, :, :, period] # shape: [batch_size, num_nodes, feature_dim]. 이런 식으로 반복문 돌리면 period와 같은 차원은 없어짐(축소)
            H_current = self._base_tgcn(X_current, edge_index, edge_weight, H_previous)
            H_previous = H_current
            H_sequence_outputs.append(probs[period] * H_current)

        H_accum = sum(H_sequence_outputs)

        return H_accum

In [6]:
import torch
import torch.nn as nn
import pandas as pd
from torch_geometric.data import Batch, Data

class EntityEmbedding(torch.nn.Module):
    def __init__(self, col_dims: list, col_list: list):
        '''
        Args:
            cat_dims: 변수별 범주의 개수 리스트
            col_list: 원본 데이터프레임에서 변수들의 순서 왼->오
        '''
        super().__init__()
        self.col_dims = col_dims
        self.col_list = col_list
        # proj_dim must be calculated with the final, corrected col_dims,
        # so we do it here with the provided ones, but it will be re-calculated in the test block
        self.proj_dim = int(max(self.col_dims)**0.5) if self.col_dims else 1

        self.embs = nn.ModuleList([
            nn.Embedding(num_categories, self.proj_dim)
            for num_categories in self.col_dims
        ])

    def forward(self, x_cats):
        '''
        This forward method assumes x_cats is a 2-D tensor of shape [N, F]
        where N is batch_size and F is number of features.
        The current data pipeline produces a different shape, so this is not used.
        '''
        # This logic is for a different data shape and is preserved for potential future use.
        x_cats = x_cats.long()
        outs = []
        for i, emb in enumerate(self.embs):
            out = emb(x_cats[:, i])
            outs.append(out)
        outs_tensor = torch.stack(outs, dim = 1)
        return outs_tensor


    """
    EntityEmbeddingBatch3

    이 모듈은 다수의 범주형 변수(예: TEDS-D의 72개 변수)를 하나의 전역 임베딩
    테이블로 처리하기 위한 전용 임베딩 레이어이다. 기존 방식처럼 변수마다
    개별적으로 nn.Embedding 레이어를 생성하고 for-loop로 순차적으로 호출하면
    GPU 커널 호출이 매우 많아져 학습 속도가 저하된다. 본 모듈은 모든 변수의
    범주(category)를 하나의 전역 vocabulary 공간으로 합쳐 단일 Embedding 레이어를
    사용함으로써 연산 효율을 크게 향상시킨다.

    -------------------------------------------------------------------------
    왜 이런 방식이 필요한가?
    -------------------------------------------------------------------------
    - 범주형 변수가 많을수록(예: 72개) 변수별 개별 임베딩 레이어를 호출하는 것은
      비효율적이다. 각 변수마다 embedding lookup을 수행하면 forward마다 72개의
      GPU 커널 호출이 발생하여 병목이 된다.
    - 모든 범주를 하나의 “전역 인덱스(global index)” 공간에 배치하면nn.Embedding을
      단 한 번만 호출하면 되므로 연산량이 대폭 감소한다.
    - 더 빠르고 간결한 구조이며, 파라미터 수는 기존 방식과 동일하다.
      (기존에는 여러 임베딩 테이블을 합쳐놓았던 것을 물리적으로 하나로 묶은 것뿐)

    -------------------------------------------------------------------------
    어떻게 동작하는가?
    -------------------------------------------------------------------------
    1. col_dims: 각 변수(컬럼)의 고유 카테고리 개수 리스트를 입력 받는다.
       예: [3, 6, 7, 4, …]

    2. offsets 계산:
       각 변수가 전역 임베딩 테이블에서 시작하는 위치를 누적합으로 계산한다.
       예: col_dims = [3, 6, 7] → offsets = [0, 3, 9]

    3. batch(로컬 인덱스):
       입력 batch는 (batch_size, num_features) 형태이며, 각 값은 해당 변수 안에서의
       로컬 카테고리 인덱스(0~V_j-1)이다.

    4. 전역 인덱스로 변환:
       glob_batch = batch + offsets
       브로드캐스팅으로 각 변수 위치에 맞는 offset이 자동으로 더해진다.

    5. embedding lookup:
       전역 인덱스 텐서를 단일 Embedding 레이어에 넣어
       (batch_size, num_features, embedding_dim) 형태의 임베딩을 얻는다.

    -------------------------------------------------------------------------
    입력
    -------------------------------------------------------------------------
    batch : torch.Tensor
        shape = [BATCH_SIZE, NUM_VAR]
        각 요소는 "해당 변수에서의 로컬 카테고리 인덱스"를 의미한다.

    -------------------------------------------------------------------------
    출력
    -------------------------------------------------------------------------
    torch.Tensor
        shape = [BATCH_SIZE, NUM_VAR, embedding_dim]
        모든 범주형 변수가 단일 임베딩 테이블에서 변환된 임베딩 벡터.

    -------------------------------------------------------------------------
    주요 장점
    -------------------------------------------------------------------------
    - 기존 방식(변수별 72개의 embedding layer + for-loop)보다 훨씬 빠름.
    - 파라미터 수는 동일하지만 연산량과 GPU kernel 호출 횟수가 압도적으로 감소함.
    - 유지보수 간단: embedding layer 하나만 관리하면 됨.
    - offsets와 col_dims는 register_buffer로 관리되어 장치 이동(to(device))과
      state_dict 저장 시 안정적으로 포함된다.
    """


class EntityEmbeddingBatch3(nn.Module):
    def __init__(self, col_dims, embedding_dim=32):
        super().__init__()
        col_dims = torch.as_tensor(col_dims, dtype=torch.long)

        offsets = torch.zeros_like(col_dims)
        offsets[1:] = torch.cumsum(col_dims[:-1], dim=0) # 누적합 계산

        # 학습 파라미터는 아니지만, 모델과 함께 device를 따라가야 하므로 buffer로 등록
        self.register_buffer("offsets", offsets)
        self.register_buffer("col_dims", col_dims)

        total_dim = int(col_dims.sum().item())  # 전체 카테고리 수
        self.embedding_layer = nn.Embedding(total_dim, embedding_dim)


    def forward(self, batch: torch.Tensor): # batch shape: [BATCH_SIZE, NUM_VAR] (=[32, 72])
        batch = batch.long()
        glob_batch = batch + self.offsets # type: ignore
        return self.embedding_layer(glob_batch) # shape: [BATCH_SIZE, NUM_VAR, FEATURE_DIM]

In [7]:
import torch
import torch.nn as nn
import sys
import os

def to_temporal(x_tensor: torch.Tensor, # shape: [batch_size, num_var, feature_dim] (=[32, 72, 25])
                ad_col_index: list,
                dis_col_index: list,
                LOS: torch.Tensor,
                device,
                max_los=37):
    batch_size, _, num_features = x_tensor.shape
    num_nodes = len(ad_col_index)

    ad_idx_t = torch.tensor(ad_col_index, device=device)
    dis_idx_t = torch.tensor(dis_col_index, device=device)

    ad_tensor = torch.index_select(x_tensor, dim=1, index=ad_idx_t)
    dis_tensor = torch.index_select(x_tensor, dim=1, index=dis_idx_t)

    # Create a temporal mask based on LOS
    los_mask = torch.arange(max_los, device=device).unsqueeze(0).unsqueeze(0).unsqueeze(0) < LOS.unsqueeze(1).unsqueeze(1).unsqueeze(1)
    los_mask = los_mask.expand(batch_size, num_nodes, num_features, max_los)

    # Create the temporal tensor
    temporal_tensor = torch.where(los_mask, ad_tensor.unsqueeze(-1), dis_tensor.unsqueeze(-1))

    return temporal_tensor



class A3TGCNCat1(nn.Module):
    '''
    tensor 연산 위주로 수행하는 모델
    '''
    def __init__(self, batch_size, col_list, col_dims, embedding_dim, hidden_channel):
        '''
        Args:
            col_info(list): [col_dims, col_list]
                            col_list(list): 데이터에서 나타나는 변수의 순서
                            col_dims(list): 각 변수 별 범주의 개수, 순서는 col_list를 따라야 함
            embedding_dim(int): 엔티티 임베딩 후의 차원
            hidden_channel(int): TGCN의 hidden channel
        '''
        super().__init__()
        self.batch_size = batch_size
        self.col_dims = col_dims
        self.col_list = col_list
        self.hidden_channel = hidden_channel

        # EntityEmbedding 레이어 정의
        self.entity_embedding_layer = EntityEmbeddingBatch3(col_dims=self.col_dims, embedding_dim=embedding_dim)

        # A3TGCN2 레이어 정의
        a3tgcn_input_channel = embedding_dim

        self.a3tgcn_layer = A3TGCN2(in_channels=a3tgcn_input_channel,
                        out_channels=hidden_channel,
                        periods=37,
                        batch_size=batch_size)

        # 분류기 레이어 정의
        self.classifier_b = nn.Sequential(
            nn.Linear(hidden_channel, hidden_channel * 2),
            nn.ReLU(),
            nn.Linear(hidden_channel * 2, 2)
        )

    def forward(self, ad_col_index, dis_col_index, x_batch: torch.Tensor, LOS_batch: torch.Tensor, template_edge_index: torch.Tensor, device):
        '''
        Args:
            batch(torch.Tensor): X, y만 정의되어 있는 Data 객체,
            template_edge_index(torch.Tensor): edge_index는 동일하므로 template_edge_index로 한꺼번에 전달
        '''
        x_embedded = self.entity_embedding_layer(x_batch)
        x = to_temporal(x_embedded, ad_col_index, dis_col_index, LOS_batch, device)
        after_GNN = self.a3tgcn_layer(x, template_edge_index) # [32, 60, 32]

        # global mean pooling [32, 60, 32] -> [32, 32]
        mean_pooled = torch.mean(after_GNN, dim=1)

        return self.classifier_b(mean_pooled)


In [8]:
'''
결국 pyg Data / Batch를 생성해서 저장 또는 로드할 필요가 없었고, 오히려 불편함, 저장공간 낭비였음
37개의 타임스탬프를 미리 만들어 저장하는 것은 원본 데이터의 37배에 달하는 용량으로 뻥튀기가 되는 것
우리는 어차피 필요한 게 1. 입소 시, 2. 퇴소 시, 3. 제로 패딩 이기 때문에
1,2만 들고 엔티티 임베딩까지 한 뒤에
모델 입력하기 직전에 시계열 데이터로 변환하면 됨

이 모든 걸 텐서로만 진행할 수 있음 + pyg Data / Batch를 쓰면 기본 알고리즘 때문에 쉐입이 이상해지던가 객체를 생성해서 저장하기 때문에 용량이 더 커짐


독스트링은 LLM에 의해 생성됨
'''
import os
import torch
import pandas as pd

from torch.utils.data import Dataset, DataLoader

class TEDSTensorDataset(Dataset):
    """
    TEDS (Temporal Embedding Deep Sequence) 모델을 위한 PyTorch Dataset.

    원본 데이터를 pyg.Data/Batch 대신 **순수 PyTorch Tensor** 형태로 변환하고 저장하여,
    저장 공간 낭비와 불필요한 객체 생성 오버헤드를 방지합니다. 데이터에는 입소(Admission)
    시점과 퇴소(Discharge) 시점 데이터만이 포함됩니다.

    Attributes:
        root (str): 데이터 저장 및 로드 경로의 루트 디렉토리.
        processed_tensor (torch.Tensor): 전처리된 최종 데이터 텐서 (입력 데이터 + 레이블).
                                         Shape: (num_samples, num_features).
                                         DataLoader에게 이걸 X, y로 나누어 전달하게 됨
        col_info (tuple[list[int], list[int]]): 컬럼 정보.
                                                (입소 시점 컬럼 인덱스 리스트, 퇴소 시점 컬럼 인덱스 리스트)
        LOS (pandas.Series): Length of Stay (재원 일수) 정보.
    """
    def __init__(self, root):
        """
        TEDSTensorDataset의 생성자.

        데이터 경로 설정, 디렉토리 생성 후, 전처리된 데이터를 로드하거나
        새로운 전처리 과정을 수행하여 데이터를 메모리에 로드합니다.

        Args:
            root: 데이터가 위치할 루트 디렉토리 경로.
        """
        super().__init__()
        self.root = root
        self.raw_dir = os.path.join(root, "raw")
        if not os.path.exists(self.raw_dir):
            os.mkdir(self.raw_dir)

        self.process_dir = os.path.join(root, 'process')
        if not os.path.exists(self.process_dir):
            os.mkdir(self.process_dir)

        processed_data_path = os.path.join(self.process_dir, "processed_data.pt")
        if os.path.exists(processed_data_path):
            print("저장되어 있는 전처리된 데이터가 있습니다. 해당 데이터를 불러오는 중..")
            self.processed_tensor, self.col_info, self.LOS = torch.load(processed_data_path, weights_only=False)
            print("불러오기 완료")
        else:
            print("저장되어 있는 전처리된 데이터가 없으므로, 전처리 과정을 진행합니다. 전처리된 데이터는 저장됩니다.")
            print(f"저장 경로: {processed_data_path}")
            processed_data = self.process
            print("전처리 완료")
            self.processed_tensor, self.col_info, self.LOS = processed_data
            print("불러오기 완료")
            torch.save(processed_data, processed_data_path)
            print("전처리된 데이터 저장 완료")

    def __getitem__(self, index):
        """
        주어진 인덱스에 해당하는 하나의 샘플과 레이블을 반환합니다.

        Args:
            index: 데이터 셋 내의 샘플 인덱스.

        Returns:
            (input_tensor, y_label): 입력 텐서와 레이블 텐서의 튜플.
        """
        input_tensor = self.processed_tensor[index, :-1]
        y_label = self.processed_tensor[index, -1]
        los = self.LOS[index]
        return input_tensor, y_label, los

    def __len__(self):
        """
        데이터 셋의 전체 샘플 개수를 반환합니다.

        Returns:
            데이터 셋의 크기 (샘플 개수).
        """
        return self.processed_tensor.shape[0]

    @property
    def process(self):
        """
        원본 데이터를 읽고 전처리 과정을 수행합니다.

        전처리 단계:
        1. CSV 파일 로드 ('missing_corrected.csv')
        2. 'LOS' (Length of Stay) 컬럼 분리 및 제거
        3. 레이블 ('REASONb') 정리 (organize_labels)
        4. 컬럼 정보 추출 (get_col_info)
        5. Pandas DataFrame을 PyTorch Tensor로 변환 (df_to_tensor)

        Raises:
            ValueError: 원본 데이터에 'LOS' 또는 'REASONb' 컬럼이 없을 경우 발생.

        Returns:
            (df_tensor, col_info, LOS): 전처리된 데이터 튜플.
        """
        data_path = os.path.join(self.raw_dir, 'missing_corrected.csv')
        df = pd.read_csv(data_path)

        # los 따로 빼기
        if 'LOS' in df.columns:
            LOS = df['LOS']
            LOS = df_to_tensor(LOS)
            df = df.drop('LOS', axis=1)
        else:
            raise ValueError('raw data에서 LOS 데이터를 찾을 수 없습니다.')

        if 'REASONb' not in df.columns:
            raise ValueError('raw data에서 REASONb 데이터를 찾을 수 없습니다.')

        # label_organize
        df = organize_labels(df)
        # df to tensor
        df_tensor = df_to_tensor(df)

        # get col infos, list of (col_list, col_dims, ad_col_index, dis_col_index)
        # ad_col_index, dis_col_index는 다음과 같음 integer position of admission col, discharge col
        df = df.drop("REASONb", axis=1)
        col_info = get_col_info(df)

        # processed_data는 (tensor, col_info, LOS)형태
        # LOS는 pd.Series임
        # col_info는 다음과 같음 (col_list, col_dims, ad_col_index, dis_col_index)
        return df_tensor, col_info, LOS # -> self.process하면 tuple로 반환될 것


In [9]:
import os
import numpy as np
from tqdm.notebook import tqdm
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

class EarlyStopper:
    def __init__(self, patience=10, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_validation_loss = float('inf')
        self.early_stop = False

    def __call__(self, validation_loss):
        if validation_loss < self.best_validation_loss - self.min_delta:
            self.best_validation_loss = validation_loss
            self.counter = 0
        elif validation_loss > self.best_validation_loss + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                print(f"🛑 Early stopping triggered after {self.counter} epochs without improvement.")

        return self.early_stop

def train(model, dataloader, criterion, optimizer, edge_index, device, scaler, ad_col_index, dis_col_index):
    model.train()
    running_loss = 0.0
    for x_batch, y_batch, los_batch in tqdm(dataloader, desc="train_process", leave=True):
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        los_batch = los_batch.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            result = model(
                ad_col_index,
                dis_col_index,
                x_batch,
                los_batch,
                edge_index,
                device
            )
            loss = criterion(result, y_batch)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * x_batch.size(0)

        epoch_loss = running_loss / len(dataloader.dataset)
    return epoch_loss

def evaluate(model, val_dataloader, criterion, device, ad_col_index, dis_col_index, edge_index, scaler):
    model.eval()
    running_loss = 0.0
    total_correct = 0
    total_samples = 0

    all_targets = []
    all_predictions = []
    all_scores = []

    with torch.no_grad():
        for x_batch, y_batch, los_batch in tqdm(val_dataloader, desc="eval_process", leave=True):
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            los_batch = los_batch.to(device)

            with torch.amp.autocast('cuda'):
                result = model(
                    ad_col_index,
                    dis_col_index,
                    x_batch,
                    los_batch,
                    edge_index,
                    device
                )
                loss = criterion(result, y_batch)

            running_loss += loss.item() * x_batch.size(0)

            _, predicted = torch.max(result, 1)

            all_targets.append(y_batch.cpu().numpy())
            all_predictions.append(predicted.cpu().numpy())
            all_scores.append(result.cpu().numpy())

            total_correct += (predicted == y_batch).sum().item()
            total_samples += y_batch.size(0)

    all_targets = np.concatenate(all_targets)
    all_predictions = np.concatenate(all_predictions)
    all_scores = np.concatenate(all_scores)

    epoch_loss = running_loss / len(val_dataloader.dataset)
    epoch_accuracy = total_correct / total_samples

    epoch_precision = precision_score(all_targets, all_predictions, average='macro', zero_division=0)
    epoch_recall = recall_score(all_targets, all_predictions, average='macro', zero_division=0)
    epoch_f1 = f1_score(all_targets, all_predictions, average='macro', zero_division=0)

    try:
        epoch_auc = roc_auc_score(all_targets, all_scores, multi_class='ovr')
    except ValueError:
        print("Warning: AUC score could not be calculated (requires multiple classes or specific binary format).")
        epoch_auc = 0.0

    return epoch_loss, epoch_accuracy, epoch_precision, epoch_recall, epoch_f1, epoch_auc

def save_checkpoint(epoch, model, optimizer, scheduler, best_loss, filename):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_loss': best_loss,
    }
    torch.save(checkpoint, filename)

def device_set():
    device = torch.device('cpu')

    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.mps.is_available():
        os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
        device = torch.device('mps')

    print(f'Using device: {device}')

    return device

In [ ]:
root = '/content/drive/MyDrive/Colab Notebooks/VAM_with_GNN/data_tensor_cache'
EPOCH = 100
scheduler_patience = 10
early_stopping_patience = 15
model_path = os.path.join(root, '/model/best_model.pt')
device = device_set()
num_workers = 12


BATCH_SIZE = 128 # GPU 메모리가 허용한다면 64, 128 등으로 늘려보세요.
embedding_dim = 32
hidden_channel = 64 ###이게 좀 걸림


from torch.optim.lr_scheduler import ReduceLROnPlateau

dataset = TEDSTensorDataset(root)

train_dataloader, val_dataloader, test_dataloader = train_test_split_customed(dataset, batch_size=BATCH_SIZE, num_workers=num_workers)

col_list, col_dims, ad_col_index, dis_col_index = dataset.col_info

model = A3TGCNCat1(batch_size=BATCH_SIZE, col_list=col_list,
                    col_dims=col_dims, embedding_dim=embedding_dim, hidden_channel=hidden_channel)

model.to(device)
print("compiling model...")
model = torch.compile(model=model, mode="max-autotune")
print("compile finished")

mi_dict_path = os.path.join(root, 'data', 'mi_dict_static.pickle')
edge_index = mi_edge_index_batched(batch_size=BATCH_SIZE,
                                        mi_dict_path=mi_dict_path,
                                        top_k=6,
                                        return_edge_attr=False)
edge_index = edge_index.to(device)

counter = 0

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

scheduler = ReduceLROnPlateau(optimizer, "min", patience=10)

early_stopper = EarlyStopper(patience=early_stopping_patience)

# Initialize GradScaler for Automatic Mixed Precision
scaler = torch.amp.GradScaler('cuda')

for epoch in tqdm(range(EPOCH)):
    train_loss = train(model, train_dataloader, criterion, optimizer, edge_index, device, scaler, ad_col_index, dis_col_index)

    result = evaluate(model, val_dataloader, criterion, device, ad_col_index, dis_col_index, edge_index, scaler)
    val_loss, val_accuracy, val_precision, val_recall, val_f1, val_auc = result

    scheduler.step(val_loss)

    if val_loss < early_stopper.best_validation_loss:
        print(f"🎉 New best validation loss: {val_loss:.4f}. Saving model...")

        best_val_loss = val_loss

        file_name = f"best_model_epoch_{epoch+1}_loss_{best_val_loss:.4f}.pth"

        save_checkpoint(epoch + 1,
                        model,
                        optimizer,
                        scheduler,
                        best_val_loss,
                        file_name)

    should_stop = early_stopper(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    print(f"\n[Epoch {epoch+1}/{EPOCH}]")
    print(f"  [Train] LR: {current_lr:.6f} | Loss: {train_loss:.4f}")
    print(f"  [Valid] Loss: {val_loss:.4f} | Acc: {val_accuracy:.4f}, Prec: {val_precision:.4f}, Rec: {val_recall:.4f}, F1: {val_f1:.4f}, AUC: {val_auc:.4f}")

    if should_stop:
        print("\n--- Early Stopping activated. Learning terminated. ---")
        break

print("\n--- 학습 완료 ---")

Using device: cuda
저장되어 있는 전처리된 데이터가 있습니다. 해당 데이터를 불러오는 중..
불러오기 완료
Train Set Size: 975897
Test Set Size: 209120
Using device: cuda
compiling model...
compile finished


  0%|          | 0/100 [00:00<?, ?it/s]

train_process:   0%|          | 0/7625 [00:00<?, ?it/s]

W1117 08:02:47.671000 1430 torch/_inductor/utils.py:1436] [9/0] Not enough SMs to use max_autotune_gemm mode
AUTOTUNE addmm(128x2, 128x128, 128x2)
strides: [0, 1], [128, 1], [1, 128]
dtypes: torch.float16, torch.float16, torch.float16
  bias_addmm 0.0072 ms 100.0% 
  addmm 0.0092 ms 77.8% 
SingleProcess AUTOTUNE benchmarking takes 0.0870 seconds and 0.0002 seconds precompiling for 2 choices


In [ ]:
import torch
from torch.profiler import profile, record_function, ProfilerActivity

# GPU를 사용한다면 CUDA 활동을 포함합니다.
activities = [
    ProfilerActivity.CPU,
    ProfilerActivity.CUDA
]

# Chrome Trace 형식으로 저장
def trace_handler(p):
    output_dir = './profiler_results'
    import os
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    p.export_chrome_trace(f"{output_dir}/trace_{p.step_num}.json")

# 학습 루프 예시
N_STEPS = 10

# 1. 스케줄 정의: 1 대기, 1 예열, 3 활성화, 1 반복
schedule = torch.profiler.schedule(
    wait=1,
    warmup=1,
    active=3,
    repeat=1
)

with profile(
    activities=activities,
    schedule=schedule,
    on_trace_ready=trace_handler, # 결과를 파일로 저장
    record_shapes=True, # 텐서 크기 기록
    profile_memory=True # 메모리 사용량 기록
) as p:
    # `train_dataloader`에서 배치를 가져오기 위한 이터레이터를 생성합니다.
    # 프로파일링 루프가 데이터로더를 소진하지 않도록 필요에 따라 리셋될 수 있습니다.
    data_iterator = iter(train_dataloader)

    for step in range(N_STEPS):
        try:
            x_batch, y_batch, los_batch = next(data_iterator)
        except StopIteration:
            # 데이터로더가 소진되면, 새로운 이터레이터를 생성합니다.
            data_iterator = iter(train_dataloader)
            x_batch, y_batch, los_batch = next(data_iterator)

        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        los_batch = los_batch.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            with record_function("model_forward"):
                result = model(
                    ad_col_index,
                    dis_col_index,
                    x_batch,
                    los_batch,
                    edge_index,
                    device
                )
                loss = criterion(result, y_batch)

        with record_function("backward_and_optimize"):
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        print(f"현재 단계: {step}, 프로파일러 상태: {p.current_action}")

        # 3. 다음 단계를 프로파일러에 알림
        p.step()

In [ ]:
# 모든 단계 완료 후 (with 블록 밖에서)
# 각 연산의 평균 시간을 기준으로 통계를 출력합니다.
print(p.key_averages().table(sort_by="cuda_time_total", row_limit=10))